In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
import plotnine as nine

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('data/31mar2026.tsv', sep='\t', low_memory=False)

In [4]:
# Convert the date column to datetime
df['Time (UTC)'] = pd.to_datetime(df['Time (UTC)'], format='%Y-%m-%d.%H:%M')

In [5]:
features = [
    'Time (UTC)',
    'HREG_T_OP2',
    'HREG_EFFECTIVE_TF',
    'HREG_EFFECTIVE_PF',
    'HREG_T_FRS',
    'HREG_T_SPLY_LTO',
    'HREG_T_SPLY',
    'HREG_T_WST',
    'HREG_T_EXT',
    'HREG_HUM_EXT',
    'HREG_AI1_RES',
    'HREG_AI3_RES',
    'HREG_AI4_RES',
    'HREG_AI5_RES',
    'HREG_AI6_RES',
    'HREG_LTO_N_SPLY',
    'HREG_LTO_N_EXT',
    'HREG_RH_MEAN',
    'HREG_OUTPUT'
]

df = df[features]

In [6]:
df_keep = df.copy()

In [7]:
df.dtypes

Time (UTC)           datetime64[us]
HREG_T_OP2                      str
HREG_EFFECTIVE_TF               str
HREG_EFFECTIVE_PF               str
HREG_T_FRS                      str
HREG_T_SPLY_LTO                 str
HREG_T_SPLY                     str
HREG_T_WST                      str
HREG_T_EXT                      str
HREG_HUM_EXT                    str
HREG_AI1_RES                    str
HREG_AI3_RES                    str
HREG_AI4_RES                    str
HREG_AI5_RES                    str
HREG_AI6_RES                    str
HREG_LTO_N_SPLY                 str
HREG_LTO_N_EXT                  str
HREG_RH_MEAN                    str
HREG_OUTPUT                     str
dtype: object

In [8]:
df

,Time (UTC),HREG_T_OP2,HREG_EFFECTIVE_TF,HREG_EFFECTIVE_PF,HREG_T_FRS,HREG_T_SPLY_LTO,HREG_T_SPLY,HREG_T_WST,HREG_T_EXT,HREG_HUM_EXT,HREG_AI1_RES,HREG_AI3_RES,HREG_AI4_RES,HREG_AI5_RES,HREG_AI6_RES,HREG_LTO_N_SPLY,HREG_LTO_N_EXT,HREG_RH_MEAN,HREG_OUTPUT
0,2013-05-05 12:47:00,-,-,-,"10,6","18,2","18,8",12,"21,4",27,26,"26,2","59,4",524,2640,70,86,27,58
1,2013-05-05 12:53:00,-,-,-,"10,6","15,9","17,2","13,9","21,4",27,25,22,"59,4",516,2640,49,69,27,2
2,2013-05-05 12:58:00,-,-,-,"10,6","11,7","13,9","18,6","21,3",26,25,"21,9","59,4",503,2640,0,0,27,0
3,2013-05-05 13:03:00,-,-,-,"10,6",11,"13,1","19,1","21,5",26,25,"21,9","59,4",495,2640,0,0,27,0
4,2013-05-05 13:08:00,-,-,-,"10,4","10,7","12,8","19,3","21,5",26,25,"21,9","59,4",490,2640,0,0,27,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1226109,2026-03-31 11:53:00,"21,3",41,45,"10,6","18,3","18,6","12,8","19,7",31,29,0,"22,5",0,470,84,75,40,100
1226110,2026-03-31 11:58:00,"21,3",41,45,"10,5","18,2","18,6","12,8","19,8",31,29,0,"22,5",0,472,83,75,40,100
1226111,2026-03-31 12:03:00,"21,3",41,45,"10,5","18,2","18,5","12,6","19,7",31,28,0,"22,5",0,462,84,76,40,100
1226112,2026-03-31 12:08:00,"21,3",41,45,"10,2","18,1","18,6","12,5","19,7",31,28,0,"22,6",0,462,83,76,40,100


In [9]:
numeric_cols = [col for col in df.columns if col != 'Time (UTC)']

df[numeric_cols] = df[numeric_cols].apply(lambda x: x.astype(str).str.replace(',', '.', regex=False))
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [10]:
df

,Time (UTC),HREG_T_OP2,HREG_EFFECTIVE_TF,HREG_EFFECTIVE_PF,HREG_T_FRS,HREG_T_SPLY_LTO,HREG_T_SPLY,HREG_T_WST,HREG_T_EXT,HREG_HUM_EXT,HREG_AI1_RES,HREG_AI3_RES,HREG_AI4_RES,HREG_AI5_RES,HREG_AI6_RES,HREG_LTO_N_SPLY,HREG_LTO_N_EXT,HREG_RH_MEAN,HREG_OUTPUT
0,2013-05-05 12:47:00,NaN,NaN,NaN,10.6,18.2,18.8,12.0,21.4,27.0,26.0,26.2,59.4,524.0,2640.0,70.0,86.0,27.0,58.0
1,2013-05-05 12:53:00,NaN,NaN,NaN,10.6,15.9,17.2,13.9,21.4,27.0,25.0,22.0,59.4,516.0,2640.0,49.0,69.0,27.0,2.0
2,2013-05-05 12:58:00,NaN,NaN,NaN,10.6,11.7,13.9,18.6,21.3,26.0,25.0,21.9,59.4,503.0,2640.0,0.0,0.0,27.0,0.0
3,2013-05-05 13:03:00,NaN,NaN,NaN,10.6,11.0,13.1,19.1,21.5,26.0,25.0,21.9,59.4,495.0,2640.0,0.0,0.0,27.0,0.0
4,2013-05-05 13:08:00,NaN,NaN,NaN,10.4,10.7,12.8,19.3,21.5,26.0,25.0,21.9,59.4,490.0,2640.0,0.0,0.0,27.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1226109,2026-03-31 11:53:00,21.3,41.0,45.0,10.6,18.3,18.6,12.8,19.7,31.0,29.0,0.0,22.5,0.0,470.0,84.0,75.0,40.0,100.0
1226110,2026-03-31 11:58:00,21.3,41.0,45.0,10.5,18.2,18.6,12.8,19.8,31.0,29.0,0.0,22.5,0.0,472.0,83.0,75.0,40.0,100.0
1226111,2026-03-31 12:03:00,21.3,41.0,45.0,10.5,18.2,18.5,12.6,19.7,31.0,28.0,0.0,22.5,0.0,462.0,84.0,76.0,40.0,100.0
1226112,2026-03-31 12:08:00,21.3,41.0,45.0,10.2,18.1,18.6,12.5,19.7,31.0,28.0,0.0,22.6,0.0,462.0,83.0,76.0,40.0,100.0


In [11]:
def label_hru(df):
    condition = (
        (df['HREG_OUTPUT'] == 100) &
        (df['HREG_LTO_N_SPLY'] > 70) &
        (df['HREG_LTO_N_EXT'] > 70)
    )
    df['HRU_LABEL'] = condition.apply(lambda x: 0 if x else 1)
    return df

df = label_hru(df)

In [13]:
df['HRU_LABEL'].value_counts()

HRU_LABEL
0    829702
1    396412
Name: count, dtype: int64

In [ ]:
df_filtered = df[df['Time (UTC)'].between('2026-03-01', '2026-03-31')]
df_filtered.size
df_filtered.dropna()

In [22]:
X = df[numeric_cols]
y = df['HRU_LABEL']

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

dtc = DecisionTreeClassifier()
dtc.fit(X_train, y_train)

y_pred = dtc.predict(X_test)

f1_score(y_test, y_pred)

0.9883674119798044

In [35]:
## removing the three columns we used to make the column
X = X.drop(['HREG_LTO_N_SPLY','HREG_LTO_N_EXT', 'HREG_OUTPUT'], axis=1)

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

dtc = DecisionTreeClassifier()
dtc.fit(X_train, y_train)

y_pred = dtc.predict(X_test)

f1_score(y_test, y_pred)

0.9884140741952122